# Planet Radius Physical Plausibility Check

**Purpose**: Follow on from the odd-even check output; for each candidate, compute the planet's absolute radius and reject false positives that are "depth-stable but the body is too large" (stellar companions / brown dwarfs) — signals that the odd-even check cannot catch.

**Physical Principle**:
- Transit depth only gives the radius ratio: $R_p/R_\star=\sqrt{\delta}$ ($\delta$ = depth fraction)
- To obtain the absolute planet radius, multiply by the host star's true radius: $R_p=\sqrt{\delta}\times R_\star$
- There is a physical ceiling on body radius (degeneracy pressure): above Jupiter's mass, radius essentially stops growing; the planetary radius upper limit is about 2-2.5 Rjup. Anything larger is a brown dwarf or star.
- **M dwarfs** have low-mass disks and almost cannot host giant planets far exceeding Jupiter; their upper limit is stricter.

**Workflow**:
```
1. astroquery -> TIC/Gaia    Fetch host star radius R* and effective temperature Teff
2. Compute radius             Rp = sqrt(depth) x R*   (converted to Jupiter/Earth radius units)
3. Classify spectral type     From Teff: O/B/A/F/G/K/M
4. Decision                   Rp > 2.5 Rjup (M dwarf: 2.0) -> implausible; no R* available -> undetermined
```

**Input**: Prefers the odd-even check output `odd_even_result.csv`, falls back to Person1's `master_candidates.csv`.
**Output**: New boolean column `planet_radius_plausible` (plus diagnostic columns), saved as `radius_result.csv`. Person1's original fields are never modified.

## 0. Install Dependencies

In [ ]:
%pip install -q numpy pandas astropy astroquery

## 1. Configuration and Data Loading

Preferentially read the output from the previous step (odd-even check), so that results from all three validation checks accumulate in the same table.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
from collections import Counter

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 220)

# ---------------- Physical Constants and Thresholds ----------------
R_SUN_TO_RJUP   = 9.7311      # 1 solar radius = 9.73 Jupiter radii
R_JUP_TO_REARTH = 11.2089     # 1 Jupiter radius = 11.21 Earth radii
MAX_RP_RJUP_GENERAL = 2.5     # general upper limit: >2.5 Rjup enters brown dwarf / stellar companion territory
MAX_RP_RJUP_MDWARF  = 2.0     # M dwarf: giant planets extremely rare, tighter upper limit

# ---------------- Input/Output ----------------
# Prefer odd-even check output; degrade progressively to Person1's original table
_PATHS = ['odd_even_result.csv',
          'outputs/candidates_annotated.csv',
          'master_candidates.csv',
          'attachments/master_candidates.csv']
INPUT_CSV = next((p for p in _PATHS if os.path.exists(p)), _PATHS[-1])
OUTPUT_CSV = 'radius_result.csv'

# Person1's original 8 columns (used for integrity check; never modified)
PERSON1_COLS = ['Target', 'Sector', 'Period (d)', 'Period Err (d)',
                'Depth (ppm)', 'Duration (hr)', 'SNR', 'False Positive']

df = pd.read_csv(INPUT_CSV)
df_in = df.copy()                 # input snapshot for end-of-notebook integrity check
print(f'Loaded {INPUT_CSV} -> {len(df)} candidates')
print('Existing columns:', list(df.columns))
df.head()

## 2. Core Functions

- `get_stellar_params`: Query TIC for stellar radius/temperature/mass and TIC ID; returns a `note` field indicating the query result (ok / no_stellar_radius / no_tic_match / exception).
- `classify_spectral`: Coarse spectral type classification from effective temperature.
- `assess_radius`: Compute $R_p$ and judge plausibility, returning a human-readable reason.

In [ ]:
from astroquery.mast import Catalogs

_TIC_CACHE = {}

def _f(x):
    '''Safely convert a potentially masked table element to float.'''
    try:
        if x is None or np.ma.is_masked(x):
            return np.nan
        return float(x)
    except Exception:
        return np.nan

def get_stellar_params(target):
    '''Query host star parameters from TIC (integrating Gaia parameters): radius (R_sun), Teff (K), mass, TIC ID.'''
    if target in _TIC_CACHE:
        return _TIC_CACHE[target]
    out = {'tic_id': None, 'rad': np.nan, 'teff': np.nan, 'mass': np.nan, 'note': 'query_failed'}
    try:
        cat = Catalogs.query_object(str(target), catalog='TIC', radius=0.02)
        if cat is not None and len(cat) > 0:
            try:                                    # take the brightest as the host star
                order = np.argsort([_f(v) for v in cat['Tmag']])
                cat = cat[order]
            except Exception:
                pass
            r = cat[0]
            out['tic_id'] = int(r['ID'])
            out['rad']  = _f(r['rad'])
            out['teff'] = _f(r['Teff'])
            out['mass'] = _f(r['mass'])
            out['note'] = 'ok' if np.isfinite(out['rad']) else 'no_stellar_radius'
        else:
            out['note'] = 'no_tic_match'
    except Exception as e:
        out['note'] = f'exception:{type(e).__name__}'
    _TIC_CACHE[target] = out
    return out

def classify_spectral(teff):
    '''Coarse spectral type classification from effective temperature.'''
    if not np.isfinite(teff):
        return 'Unknown'
    if teff >= 30000: return 'O'
    if teff >= 10000: return 'B'
    if teff >= 7500:  return 'A'
    if teff >= 6000:  return 'F'
    if teff >= 5200:  return 'G'
    if teff >= 3700:  return 'K'
    if teff >= 2400:  return 'M'
    return 'L/T(substellar)'

def assess_radius(depth_ppm, rstar_rsun, teff):
    '''Return (plausible or pd.NA, Rp_Rjup, Rp_Rearth, spectral type, reason).'''
    spectral = classify_spectral(teff)
    d = float(depth_ppm) * 1e-6
    if not np.isfinite(d) or d <= 0:
        return pd.NA, np.nan, np.nan, spectral, 'bad_depth'
    if not np.isfinite(rstar_rsun):
        return pd.NA, np.nan, np.nan, spectral, 'no_stellar_radius'
    rp_rjup = np.sqrt(d) * rstar_rsun * R_SUN_TO_RJUP
    rp_rearth = rp_rjup * R_JUP_TO_REARTH
    max_rp = MAX_RP_RJUP_MDWARF if spectral == 'M' else MAX_RP_RJUP_GENERAL
    if rp_rjup <= max_rp:
        return True, rp_rjup, rp_rearth, spectral, 'plausible'
    reason = 'too_large_mdwarf' if spectral == 'M' else 'too_large'
    return False, rp_rjup, rp_rearth, spectral, reason

## 2.1 Single-Target Diagnostic (Run This Cell First)

Use the first candidate to verify astroquery can connect to TIC and return a stellar radius. If it returns `no_tic_match`/`exception`, the issue is usually network-related; `no_stellar_radius` means the star was found but lacks a radius entry (a data gap).

In [ ]:
_t = str(df['Target'].iloc[0])
_sp = get_stellar_params(_t)
print('Host star query:', _t, '->', _sp)
_depth = float(df['Depth (ppm)'].iloc[0])
_ok, _rj, _re, _spec, _rsn = assess_radius(_depth, _sp['rad'], _sp['teff'])
print(f'Depth={_depth}ppm  SpectralType={_spec}  Rp={_rj if np.isfinite(_rj) else "NA"} Rjup '
      f'({_re if np.isfinite(_re) else "NA"} Rearth)  Plausible={_ok}  Reason={_rsn}')
if _sp['note'] in ('no_tic_match',) or _sp['note'].startswith('exception'):
    print('>> Hint: TIC query missed or errored; check network / astroquery access.')
elif _sp['note'] == 'ok':
    print('>> Pipeline working; proceed to batch run.')

## 3. Batch Radius Validation

In [ ]:
tic_list, rstar_list, teff_list = [], [], []
spec_list, rj_list, re_list = [], [], []
plausible_list, reason_list = [], []

for i, row in df.iterrows():
    sp = get_stellar_params(row['Target'])
    ok, rj, re, spec, reason = assess_radius(row['Depth (ppm)'], sp['rad'], sp['teff'])
    tic_list.append(sp['tic_id']); rstar_list.append(sp['rad']); teff_list.append(sp['teff'])
    spec_list.append(spec); rj_list.append(rj); re_list.append(re)
    plausible_list.append(ok); reason_list.append(reason)
    rj_txt = 'NaN' if not np.isfinite(rj) else f'{rj:.2f}'
    rstar_txt = 'NA' if not np.isfinite(sp['rad']) else f"{sp['rad']:.3f}"
    print(f"[{i+1:>2}/{len(df)}] {str(row['Target']):<12} R*={rstar_txt:<6} Type={spec:<8} "
          f"Rp={rj_txt:<6}Rjup  plausible={str(ok):<5} reason={reason}")

df['TIC_ID']              = tic_list
df['Stellar_Radius_Rsun'] = rstar_list
df['Teff_K']              = teff_list
df['Spectral_Type']       = spec_list
df['Rp_Rjup']             = rj_list
df['Rp_Rearth']           = re_list
df['radius_reason']       = reason_list
df['planet_radius_plausible'] = pd.array(plausible_list, dtype='boolean')

print('\nRadius check completed ->  plausible:', int((df['planet_radius_plausible'] == True).sum()),
      '  implausible:', int((df['planet_radius_plausible'] == False).sum()),
      '  undetermined:', int(df['planet_radius_plausible'].isna().sum()))
print('Reason distribution:')
for reason, cnt in Counter(reason_list).most_common():
    print(f'   {cnt:>3}  x  {reason}')

## 4. Save Results

In [ ]:
# Integrity check: Person1's original columns must remain unchanged
_p1 = [c for c in PERSON1_COLS if c in df.columns and c in df_in.columns]
assert df[_p1].equals(df_in[_p1]), 'Person1 original data was unexpectedly modified!'
print('Original data integrity check passed.\n')

df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print('Saved ->', OUTPUT_CSV)

# Show targets flagged as implausible (suspected stellar companion / brown dwarf)
bad = df[df['planet_radius_plausible'] == False]
print(f'\nTargets flagged as implausible: {len(bad)}')
if len(bad) > 0:
    print(bad[['Target', 'Depth (ppm)', 'Stellar_Radius_Rsun', 'Spectral_Type',
               'Rp_Rjup', 'radius_reason']].to_string(index=False))
df

## 5. Notes

- `planet_radius_plausible`: `True` = radius within physical limits, `False` = Rp exceeds the upper limit (suspected star/brown dwarf), `<NA>` = undetermined because stellar radius unavailable.
- The thresholds 2.5 / 2.0 Rjup are empirical upper limits and can be adjusted in the configuration section; the literature also commonly uses ~8 Rearth (brown dwarf lower limit) as the radius ceiling for verifiable planets.
- Conservative principle: `no_stellar_radius`/`no_tic_match` are recorded as missing, never as pass.
- This step is the complementary check to odd-even validation: odd-even catches binaries with alternating depths; radius catches stellar companions whose depth is stable but whose derived body is too large.
- Output `radius_result.csv` already accumulates both odd-even + radius check results. Proceed directly to the third step: NASA Archive cross-match.